# FPGA Sketch Filter - PYNQ 硬體加速測試
這份腳本用來驗證 Vivado 導出的 `.bit` 與 `.hwh` 檔案，並在 PYNQ 平台上進行實際的硬體加速測試。

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pynq import Overlay, allocate
import time

# 1. 載入硬體
# 請確保 sketch_filter.bit 與 sketch_filter.hwh 已經上傳到跟此腳本相同的資料夾
overlay = Overlay('./../vivado_export/sketch_filter.bit')

## 步驟 1: 確認 IP 名稱與 Register Map (暫存器地圖)
由於 Vivado Block Design 以及 Vitis HLS 的版本差異，IP 的名稱或變數名稱可能會帶有 `_0` 或 `_V` 後綴。
如果後續遇到找不到屬性的報錯，請隨時回來這裡查看正確的名稱。

In [ ]:
# 印出這包 Bitstream 裡面所有可用的 IP 名稱
print("可用 IP 列表:", overlay.ip_dict.keys())

# 綁定我們的硬體加速器 IP (名稱請參考上方印出的結果，通常是 sketch_filter_0)
sketch_ip = overlay.sketch_filter_0

# 印出 Register Map，這會告訴你 rows, cols, src_mem, dst_mem 的確切呼叫名稱與記憶體位址(Offset)
print("\n--- Register Map ---")
print(sketch_ip.register_map)

## 步驟 2: 載入測試圖片並分配記憶體 (CMA)
Zynq 的特色是可以分配實體連續記憶體 (CMA)，讓硬體能直接透過 DMA / AXI Master 大量搬運資料。

In [ ]:
# 讀取測試圖片 (請準備一張 test.jpg 上傳到相同資料夾)
img = cv2.imread('test.jpg', cv2.IMREAD_GRAYSCALE)
if img is None:
    raise ValueError('讀取不到 test.jpg，請確認檔案存在')

height, width = img.shape
print(f'影像大小: {width} x {height}')

# 透過 PYNQ allocate 分配連續實體記憶體給輸入與輸出
in_buffer = allocate(shape=(height, width), dtype=np.uint8)
out_buffer = allocate(shape=(height, width), dtype=np.uint8)

# 將圖片資料放進記憶體中
np.copyto(in_buffer, img)
out_buffer[:] = 255 # 將輸出預設為全白紙張

## 步驟 3: 驅動硬體並顯示結果
透過 AXI-Lite 介面下達指令，並量測硬體計算時間。

In [ ]:
# 設定 IP 的控制暫存器 (如果報錯說找不到屬性，請參考步驟 1 印出的 Register Map 修改)
sketch_ip.register_map.rows = height
sketch_ip.register_map.cols = width

# 告訴硬體去哪裡拿資料 (給予記憶體的實體位址 Physical Address)
sketch_ip.register_map.src_mem = in_buffer.physical_address
sketch_ip.register_map.dst_mem = out_buffer.physical_address

# 開始計時
start_time = time.time()

# 啟動硬體 (AP_START)
sketch_ip.register_map.CTRL.AP_START = 1

# 輪詢等待硬體算完 (AP_DONE 變成 1)
while sketch_ip.register_map.CTRL.AP_DONE == 0:
    pass

end_time = time.time()
print(f'\n[效能] 硬體運算時間: {(end_time - start_time)*1000:.2f} 毫秒')

# 顯示結果
plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1)
plt.title('Original (Gray)')
plt.imshow(in_buffer, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Hardware Sketch Filter Output')
plt.imshow(out_buffer, cmap='gray')
plt.axis('off')
plt.show()

# 用完後務必釋放 CMA 記憶體避免崩潰
in_buffer.close()
out_buffer.close()